# Part 3 · Train the fusion Denoising-LoRA (sketch + reference together)

Loads the two frozen Condition-LoRAs from Part 2 and trains only the new Denoising-LoRA with **both conditions present**. This is the step where UniCombine's Conditional-MMDiT attention mixes the two condition streams.

- **Needs:** `lora/Condition_LoRA/sketch.safetensors` and `reference.safetensors` (Part 2, run twice) · Part 1's cache
- **Produces:** `lora/Denoising_LoRA/sketch_reference_union/` in your store
- **Cost:** more tokens per step than Part 2 (two condition images instead of one), so expect it to be the slowest part. Smoke-test first, same as Part 2; resume the same way.

In [ ]:
SMOKE      = True
STEPS      = 1000
MAX_MIN    = 150
SAVE_EVERY = 250

In [ ]:
# --- bootstrap (identical in every part): find the pipeline files, set paths ---
import os, glob, shutil, sys
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    hits = glob.glob('/kaggle/input/**/common.py', recursive=True)
    assert hits, 'Upload the pipeline/*.py files + pilot_v1.zip as a Kaggle dataset and add it as an input to this notebook.'
    SRC, WORK = os.path.dirname(hits[0]), '/kaggle/working'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    SRC, WORK = '/content/drive/MyDrive/sketchy_dataset_one_sketch/pipeline', '/content'
os.makedirs(f'{WORK}/pipeline', exist_ok=True)
for f in glob.glob(f'{SRC}/*.py'):
    shutil.copy(f, f'{WORK}/pipeline/')
sys.path.insert(0, f'{WORK}/pipeline')
from common import *
print('platform:', 'Kaggle' if ON_KAGGLE else 'Colab', '| STORE =', STORE)

In [ ]:
!pip install -q diffusers==0.33.1 transformers==4.52.4 peft==0.15.2 accelerate==1.7.0 bitsandbytes sentencepiece protobuf opencv-python-headless ipdb

In [ ]:
import torch
p = torch.cuda.get_device_properties(0)
print(p.name, round(p.total_memory / 1e9), 'GB | bf16:', torch.cuda.get_device_capability()[0] >= 8)

In [ ]:
setup_repo()   # clone UniCombine + apply the patches

In [ ]:
hf_login()   # FLUX.1-schnell is gated: accept the terms on its Hugging Face page first, then use a read token

In [ ]:
download('black-forest-labs/FLUX.1-schnell', FLUX, allow=DENOISE_PARTS)                  # ~24 GB, the 4-bit conversion happens while loading
download('ostris/FLUX.1-schnell-training-adapter', ASSIST)
CACHE = need('cache_r256.pt')
print(CACHE)

In [ ]:
CL = f'{CKPT}/Condition_LoRA'
os.makedirs(CL, exist_ok=True)
for t in ['sketch', 'reference']:
    shutil.copy(need(f'Condition_LoRA/{t}.safetensors'), f'{CL}/{t}.safetensors')
print(os.listdir(CL))
OUT = f'{STORE}/runs_smoke' if SMOKE else f'{STORE}/runs'
n = 20 if SMOKE else STEPS
!python {PIPE}/train_lowmem.py --uni_dir {UNI} --flux_dir {FLUX} --assistant_dir {ASSIST} --cache {CACHE} \
    --cond_types sketch reference --cond_lora_dir {CL} --out_name sketch_reference_union --out_dir {OUT} \
    --steps {n} --save_every {SAVE_EVERY if not SMOKE else 10} --max_minutes {MAX_MIN}

In [ ]:
src = f'{OUT}/exports/sketch_reference_union'
assert os.path.exists(f'{src}/pytorch_lora_weights.safetensors'), 'Not finished yet: run the training cell again to resume.'
if SMOKE:
    print('smoke test OK. Now set SMOKE=False and run again.')
else:
    dst = f'{STORE}/lora/Denoising_LoRA/sketch_reference_union'
    shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
    print('saved', dst)